# S1 · 두 집단 실험

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 위에서부터 각 셀을 **Shift+Enter** 로 실행하세요. 설치는 없고, 구글 계정만 있으면 됩니다.

📖 본문 학습 페이지: [S1 · 두 집단 실험](https://grow.minds.kr/textbooks/css-methods/causal/book/s1-시나리오-두-집단-실험.html)

## 1. 준비

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(처음 한 번, 수 초).
!git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


## 2. ③ 균형 점검
무작위가 **이 표본에서** 실제로 균형을 만들었는지 확인. 셋 다 차이가 없어야 한다.

In [ ]:
exp = load("exp")                       # 380 -> 366
t, c = exp[exp.cond == 1], exp[exp.cond == 0]
print(len(exp), len(t), len(c))         # 366 183 183
for v in ["age", "mil_t1"]:
    print(v, round(cohen_d(t[v], c[v]), 2))          # -0.06 / -0.01
chi2, p, df, _ = stats.chi2_contingency(pd.crosstab(exp.cond, exp.gender))
print("성별 χ²:", df, round(chi2, 2), round(p, 2))    # 2 2.28 0.32

## 3. ⑦ 주 분석, 세 길이 같은 곳으로
평균 차이 0.283 을 뒤섞기와 t검정이 각각 판정한다.

In [ ]:
obs = t.mil.mean() - c.mil.mean(); print("관찰 차이:", round(obs, 3))     # 0.283
rng = np.random.default_rng(73)
diffs = np.array([exp.mil.values[(f:=rng.permutation(exp.cond.values))==1].mean()
                  - exp.mil.values[f==0].mean() for _ in range(5000)])
print("뒤섞기 p:", round((np.abs(diffs) >= abs(obs)).mean(), 4))          # 0.0302
tt = stats.ttest_ind(t.mil, c.mil)
print("t검정:", round(tt.statistic, 2), round(tt.pvalue, 3))              # 2.21 0.028

## 4. ⑧ 강건성
효과크기·부트스트랩 구간·공변인 조정. 셋이 같은 방향이면 결론이 산다.

In [ ]:
print("d:", round(cohen_d(t.mil, c.mil), 2))                              # 0.23
tv, cv = t.mil.values, c.mil.values
rng = np.random.default_rng(73)
bd = np.array([rng.choice(tv, len(tv)).mean() - rng.choice(cv, len(cv)).mean()
               for _ in range(5000)])
print("부트 95% 구간:", np.percentile(bd, [2.5, 97.5]).round(2))          # [0.03 0.54]
b, se, p, _ = ols(exp.mil, [exp.cond, exp.mil_t1])
print("기저 조정 후:", round(b[1], 2), round(se[1], 2), round(p[1], 3))    # 0.29 0.11 0.007

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.